# Phase 4 - Layer D: Operational Scheduling Constraints

This notebook enriches the raw geometric visibility windows from Layer C with simple operational realism.

The satellite orbits and gateway coordinates are grounded in public data from the earlier layers. The operational fields in this notebook are synthetic because public Starlink gateway maintenance schedules, antenna reservations, and reconfiguration logs are not generally available.

The first operational model assumes each gateway behaves as a single dish-like resource: one active contact at a time, a fixed reconfiguration time between contacts, and reproducible blackout windows that represent maintenance or local unavailability.

## Configuration

The notebook reads only the reviewed Layer C visibility-window CSV and writes the Layer D operational assumptions and enriched windows to `data/dataset/starlink/intermediate`.

In [7]:
from pathlib import Path
import hashlib
import importlib.util
import json
import subprocess
import sys

REQUIRED_PACKAGES = ["pandas"]
missing_packages = [
    package_name
    for package_name in REQUIRED_PACKAGES
    if importlib.util.find_spec(package_name) is None
]

if missing_packages:
    print(f"Installing missing packages into this notebook kernel: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Dependency installation finished. If imports still fail, restart the kernel and run the notebook again.")

import pandas as pd

# --------- loading configuration environment ----------
# load .json config file shared by the data wrangling notebooks
CONFIG_PATH = Path("data_config.json")
data_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

DEFAULT_ANTENNA_TYPE = data_config["operational_antenna_type"]
DEFAULT_STATION_CAPACITY = data_config["operational_station_capacity"]
RECONFIGURATION_TIME_SECONDS = data_config["operational_reconfiguration_time_seconds"]
BLACKOUTS_PER_STATION = data_config["operational_blackouts_per_station"]
BLACKOUT_DURATION_MINUTES = data_config["operational_blackout_duration_minutes"]
BLACKOUT_RANDOM_SEED = data_config["operational_blackout_random_seed"]
BLACKOUT_REASON = data_config["operational_blackout_reason"]

# --------- project directory setup ----------
# intermediate datasets: only consumed by the next data wrangling notebooks
INTERMEDIATE_DIR = Path("../../data/dataset/starlink/intermediate")
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
VISIBILITY_INPUT_CSV = INTERMEDIATE_DIR / "starlink_visibility_windows.csv"
BLACKOUT_OUTPUT_CSV = INTERMEDIATE_DIR / "synthetic_gateway_blackouts.csv"
ENRICHED_OUTPUT_CSV = INTERMEDIATE_DIR / "starlink_visibility_windows_operational.csv"

print(f"Notebook Python executable: {sys.executable}")
print(f"Config path: {CONFIG_PATH.resolve()}")
print(f"Layer D intermediate data directory: {INTERMEDIATE_DIR.resolve()}")
print(f"Visibility input: {VISIBILITY_INPUT_CSV.resolve()}")
print(f"Blackout output: {BLACKOUT_OUTPUT_CSV.resolve()}")
print(f"Enriched output: {ENRICHED_OUTPUT_CSV.resolve()}")

Notebook Python executable: d:\University\PoliMi\courses\SWE2\SWE2-research-project\.venv\Scripts\python.exe
Config path: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\01_data_wrangling\data_config.json
Layer D intermediate data directory: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate
Visibility input: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_visibility_windows.csv
Blackout output: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\synthetic_gateway_blackouts.csv
Enriched output: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_visibility_windows_operational.csv


## Load Raw Visibility Windows

Layer D starts from the raw geometric visibility windows generated in Layer C. Times are parsed as UTC timestamps so blackout overlaps and availability checks are reliable.

In [8]:
# load the Layer C windows and parse the times as UTC timestamps
visibility_windows = pd.read_csv(VISIBILITY_INPUT_CSV)
visibility_windows["start_time"] = pd.to_datetime(visibility_windows["start_time"], utc=True)
visibility_windows["end_time"] = pd.to_datetime(visibility_windows["end_time"], utc=True)

print(f"Visibility windows loaded: {len(visibility_windows)}")
print(f"Unique stations: {visibility_windows['station_id'].nunique()}")
print(f"Unique satellites: {visibility_windows['norad_id'].nunique()}")
visibility_windows.head(10)

Visibility windows loaded: 73600
Unique stations: 150
Unique satellites: 983


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch
0,VIS_000001,STARLINK-1552,46076,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,48.621,2026-06-17T21:51:22Z
1,VIS_000002,STARLINK-2495,48325,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,50.028,2026-06-17T21:50:51Z
2,VIS_000003,STARLINK-2738,48668,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,66.572,2026-06-17T21:52:22Z
3,VIS_000004,STARLINK-3159,49415,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,32.655,2026-06-18T06:11:24Z
4,VIS_000005,STARLINK-3117,49435,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,50.149,2026-06-18T06:08:48Z
5,VIS_000006,STARLINK-2356,47896,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,79.033,2026-06-17T23:34:49Z
6,VIS_000007,STARLINK-3297,50159,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:05:00+00:00,300,88.099,2026-06-17T07:41:29Z
7,VIS_000008,STARLINK-3289,50173,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,38.942,2026-06-18T02:40:02Z
8,VIS_000009,STARLINK-3283,50179,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:01:00+00:00,60,28.556,2026-06-18T02:40:46Z
9,VIS_000010,STARLINK-3267,50197,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,44.213,2026-06-18T02:42:05Z


## Define Station Availability and Blackouts

Every station is available for the full Layer C horizon except during synthetic blackout windows. The blackout generator uses a hash of the station ID, blackout index, and seed, so the same inputs always produce the same blackout table.

In [9]:
def deterministic_fraction(*parts: object) -> float:
    """Create a stable pseudo-random fraction in the range [0, 1).

    Each combination of `parts` gets its own independent draw. A single seeded
    generator (e.g. `random.Random(seed)`) would not do: its values depend on how
    many draws came before, so adding or removing one station would shift the
    blackouts of every station after it.

    Parameters
    ----------
    *parts:
        Values that uniquely define the deterministic draw.

    Returns
    -------
    float
        Stable fraction derived from a SHA-256 digest.
    """
    text = "|".join(str(part) for part in parts)
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12)


def build_station_table(windows_df: pd.DataFrame) -> pd.DataFrame:
    """Build one operational resource row for each gateway station.

    Each station gets its antenna type, its capacity (contacts it can serve at
    the same time) and its reconfiguration time (minimum pause between two
    consecutive contacts while the dish re-points). 
    All stations share the same values for now; keeping them in one table lets 
    single stations differ later without touching the windows.

    Parameters
    ----------
    windows_df:
        Visibility windows containing station IDs and station names.

    Returns
    -------
    pandas.DataFrame
        Station-level operational assumptions for Layer D.
    """
    stations_df = (
        windows_df[["station_id", "station_name"]]
        .drop_duplicates()
        .sort_values(["station_id", "station_name"])
        .reset_index(drop=True)
    )
    stations_df["antenna_type"] = DEFAULT_ANTENNA_TYPE
    stations_df["station_capacity"] = DEFAULT_STATION_CAPACITY
    stations_df["reconfiguration_time_seconds"] = RECONFIGURATION_TIME_SECONDS
    return stations_df


def generate_blackout_windows(
    stations_df: pd.DataFrame,
    horizon_start: pd.Timestamp,
    horizon_end: pd.Timestamp,
    blackouts_per_station: int,
    blackout_duration_minutes: int,
    seed: int,
) -> pd.DataFrame:
    """Generate deterministic synthetic blackout windows for each station.

    Parameters
    ----------
    stations_df:
        Station-level operational table.
    horizon_start:
        Start of the operational horizon.
    horizon_end:
        End of the operational horizon.
    blackouts_per_station:
        Number of blackout windows generated for each station.
    blackout_duration_minutes:
        Fixed duration of each blackout window.
    seed:
        Stable seed included in each hash-based draw.

    Returns
    -------
    pandas.DataFrame
        Blackout table with station IDs, times, durations, and reasons.
    """
    if blackouts_per_station < 0:
        raise ValueError("blackouts_per_station cannot be negative.")
    if blackout_duration_minutes <= 0:
        raise ValueError("blackout_duration_minutes must be positive.")

    # work in seconds from the horizon start, converted back to timestamps at the end
    horizon_seconds = int((horizon_end - horizon_start).total_seconds())
    blackout_seconds = blackout_duration_minutes * 60
    if blackouts_per_station and blackout_seconds * blackouts_per_station > horizon_seconds:
        raise ValueError("Requested blackout windows do not fit inside the operational horizon.")

    # split the horizon into one equal segment per blackout, so blackouts of a station never overlap
    segment_seconds = horizon_seconds // max(1, blackouts_per_station)

    records = []
    for station in stations_df.itertuples(index=False):
        for blackout_index in range(blackouts_per_station):

            # bounds of the segment to the blackout
            segment_start_offset = blackout_index * segment_seconds
            segment_end_offset = min((blackout_index + 1) * segment_seconds, horizon_seconds)
            
            # get the last point in thime at which the bloackout can start without exceeding the segment
            latest_offset_in_segment = max(segment_start_offset, segment_end_offset - blackout_seconds)
            # get the span in which the blackout can start
            available_offset_span = max(0, latest_offset_in_segment - segment_start_offset)

            # draw the starting blackout position withint the available time range
            fraction = deterministic_fraction(seed, station.station_id, blackout_index)
            offset_seconds = segment_start_offset + int(fraction * available_offset_span)
            blackout_start = horizon_start + pd.Timedelta(seconds=offset_seconds)
            blackout_end = blackout_start + pd.Timedelta(seconds=blackout_seconds)

            records.append(
                {
                    "blackout_id": f"BLK_{station.station_id}_{blackout_index + 1:02d}",
                    "station_id": station.station_id,
                    "station_name": station.station_name,
                    "blackout_start": blackout_start.isoformat(),
                    "blackout_end": blackout_end.isoformat(),
                    "duration_seconds": blackout_seconds,
                    "reason": BLACKOUT_REASON,
                    "source": "deterministic synthetic Layer D assumption",
                }
            )

    # safeguard: if the blackout windows per station are 0 we get an error, so by default
    # we create an empty dataframe with the cloumns structure
    return pd.DataFrame.from_records(
        records,
        columns=[
            "blackout_id",
            "station_id",
            "station_name",
            "blackout_start",
            "blackout_end",
            "duration_seconds",
            "reason",
            "source",
        ],
    )


horizon_start = visibility_windows["start_time"].min().floor("min")
horizon_end = visibility_windows["end_time"].max().ceil("min")
stations = build_station_table(visibility_windows)
blackout_windows = generate_blackout_windows(
    stations,
    horizon_start,
    horizon_end,
    BLACKOUTS_PER_STATION,
    BLACKOUT_DURATION_MINUTES,
    BLACKOUT_RANDOM_SEED,
)

print(f"Operational horizon start: {horizon_start.isoformat()}")
print(f"Operational horizon end: {horizon_end.isoformat()}")
print(f"Stations with operational assumptions: {len(stations)}")
print(f"Synthetic blackout windows: {len(blackout_windows)}")
blackout_windows.head(10)

Operational horizon start: 2026-06-18T00:00:00+00:00
Operational horizon end: 2026-06-18T04:00:00+00:00
Stations with operational assumptions: 150
Synthetic blackout windows: 0


,blackout_id,station_id,station_name,blackout_start,blackout_end,duration_seconds,reason,source


## Enrich Visibility Windows

Each visibility window receives station capacity, antenna type, reconfiguration time, availability bounds, and any overlapping blackout IDs. Windows that overlap a blackout are retained but flagged as not operationally available, making the reiview more informative.

In [10]:
def find_overlapping_blackouts(
    station_id: str,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    blackouts_by_station: dict[str, pd.DataFrame],
) -> str:
    """
    Return semicolon-separated blackout IDs that overlap one visibility window.

    Parameters
    ----------
    station_id:
        Gateway station identifier for the visibility window.
    window_start:
        Visibility-window start time.
    window_end:
        Visibility-window end time.
    blackouts_by_station:
        Dictionary of blackout DataFrames keyed by station ID.

    Returns
    -------
    str
        Semicolon-separated blackout IDs, or an empty string when none overlap.
    """
    station_blackouts = blackouts_by_station.get(station_id)
    if station_blackouts is None or station_blackouts.empty:
        return ""

    overlaps = station_blackouts[
        (window_start < station_blackouts["blackout_end_ts"])
        & (window_end > station_blackouts["blackout_start_ts"])
    ]
    return ";".join(overlaps["blackout_id"].tolist())


def enrich_visibility_windows(
    windows_df: pd.DataFrame,
    stations_df: pd.DataFrame,
    blackouts_df: pd.DataFrame,
    availability_start: pd.Timestamp,
    availability_end: pd.Timestamp,
) -> pd.DataFrame:
    """Add deterministic operational scheduling constraints to visibility windows.

    Parameters
    ----------
    windows_df:
        Raw Layer C visibility windows.
    stations_df:
        Station-level operational assumptions.
    blackouts_df:
        Synthetic station blackout windows.
    availability_start:
        Start of the station availability horizon.
    availability_end:
        End of the station availability horizon.

    Returns
    -------
    pandas.DataFrame
        Enriched visibility-window table with operational constraint fields.
    """
    # turn blackout times from text into timestamps, so they can be compared with window times
    blackouts = blackouts_df.copy()
    blackouts["blackout_start_ts"] = pd.to_datetime(blackouts["blackout_start"], utc=True)
    blackouts["blackout_end_ts"] = pd.to_datetime(blackouts["blackout_end"], utc=True)
    
    # group blackouts by station, so each window only checks its own station
    blackouts_by_station = {
        station_id: group.reset_index(drop=True)
        for station_id, group in blackouts.groupby("station_id")
    }

    # add station properties (antenna, capacity, reconfiguration time) to each window
    enriched = windows_df.merge(stations_df, on=["station_id", "station_name"], how="left")
    
    # every window must have found its station in the station table
    missing_station_assumptions = enriched["antenna_type"].isna().sum()
    if missing_station_assumptions:
        raise ValueError(f"Missing station assumptions for {missing_station_assumptions} visibility windows.")

    enriched["availability_start"] = availability_start.isoformat()
    enriched["availability_end"] = availability_end.isoformat()
    # list the blackouts that overlap each window (empty text if none)
    enriched["blackout_window_ids"] = [
        find_overlapping_blackouts(row.station_id, row.start_time, row.end_time, blackouts_by_station)
        for row in enriched.itertuples(index=False)
    ]
    # a window that overlaps a blackout is kept but marked as not available
    enriched["overlaps_blackout"] = enriched["blackout_window_ids"].ne("")
    enriched["is_operationally_available"] = ~enriched["overlaps_blackout"]
    enriched["operational_constraint_source"] = "deterministic synthetic Layer D assumptions"

    # keep only the output columns, in a fixed order
    output_columns = [
        "window_id",
        "satellite_name",
        "norad_id",
        "station_id",
        "station_name",
        "start_time",
        "end_time",
        "duration_seconds",
        "max_elevation_deg",
        "source_epoch",
        "antenna_type",
        "station_capacity",
        "reconfiguration_time_seconds",
        "availability_start",
        "availability_end",
        "blackout_window_ids",
        "overlaps_blackout",
        "is_operationally_available",
        "operational_constraint_source",
    ]
    return enriched[output_columns]


enriched_windows = enrich_visibility_windows(
    visibility_windows,
    stations,
    blackout_windows,
    horizon_start,
    horizon_end,
)

print(f"Enriched visibility windows: {len(enriched_windows)}")
print(f"Operationally available windows: {int(enriched_windows['is_operationally_available'].sum())}")
print(f"Windows overlapping blackouts: {int(enriched_windows['overlaps_blackout'].sum())}")
enriched_windows.head(10)

Enriched visibility windows: 73600
Operationally available windows: 73600
Windows overlapping blackouts: 0


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1552,46076,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,48.621,2026-06-17T21:51:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-2495,48325,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,50.028,2026-06-17T21:50:51Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-2738,48668,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,66.572,2026-06-17T21:52:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-3159,49415,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,32.655,2026-06-18T06:11:24Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-3117,49435,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,50.149,2026-06-18T06:08:48Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-2356,47896,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,79.033,2026-06-17T23:34:49Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-3297,50159,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:05:00+00:00,300,88.099,2026-06-17T07:41:29Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-3289,50173,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,38.942,2026-06-18T02:40:02Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-3283,50179,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:01:00+00:00,60,28.556,2026-06-18T02:40:46Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-3267,50197,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,44.213,2026-06-18T02:42:05Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions


## Validation

Most properties of the Layer D tables are guaranteed by how they are built. The only values that can be wrong are the station properties read from `data_config.json`, so those are checked here, followed by a summary of window availability.

In [11]:
# station properties come straight from data_config.json, so check they make sense
if not str(DEFAULT_ANTENNA_TYPE).strip():
    raise ValueError("operational_antenna_type cannot be empty.")
if DEFAULT_STATION_CAPACITY < 1:
    raise ValueError("operational_station_capacity must be at least 1.")
if RECONFIGURATION_TIME_SECONDS < 0:
    raise ValueError("operational_reconfiguration_time_seconds cannot be negative.")

print("Availability counts:")
print(enriched_windows["is_operationally_available"].value_counts().to_string())
print("Preview:")
enriched_windows.head(10)

Availability counts:
is_operationally_available
True    73600
Preview:


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1552,46076,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,48.621,2026-06-17T21:51:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-2495,48325,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,50.028,2026-06-17T21:50:51Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-2738,48668,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,66.572,2026-06-17T21:52:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-3159,49415,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,32.655,2026-06-18T06:11:24Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-3117,49435,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,50.149,2026-06-18T06:08:48Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-2356,47896,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,79.033,2026-06-17T23:34:49Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-3297,50159,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:05:00+00:00,300,88.099,2026-06-17T07:41:29Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-3289,50173,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,38.942,2026-06-18T02:40:02Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-3283,50179,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:01:00+00:00,60,28.556,2026-06-18T02:40:46Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-3267,50197,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,44.213,2026-06-18T02:42:05Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions


## Save Phase 4 Outputs

The blackout table records the synthetic unavailability assumptions. The enriched visibility-window table keeps all Layer C windows while marking which ones are operationally available under those assumptions.

In [12]:
blackout_windows.to_csv(BLACKOUT_OUTPUT_CSV, index=False)
enriched_windows.to_csv(ENRICHED_OUTPUT_CSV, index=False)

print(f"Saved synthetic blackout windows to: {BLACKOUT_OUTPUT_CSV.resolve()}")
print(f"Blackout rows saved: {len(blackout_windows)}")
print(f"Saved enriched visibility windows to: {ENRICHED_OUTPUT_CSV.resolve()}")
print(f"Enriched rows saved: {len(enriched_windows)}")
enriched_windows.head(10)

Saved synthetic blackout windows to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\synthetic_gateway_blackouts.csv
Blackout rows saved: 0
Saved enriched visibility windows to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_visibility_windows_operational.csv
Enriched rows saved: 73600


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1552,46076,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,48.621,2026-06-17T21:51:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-2495,48325,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,50.028,2026-06-17T21:50:51Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-2738,48668,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,66.572,2026-06-17T21:52:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-3159,49415,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,32.655,2026-06-18T06:11:24Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-3117,49435,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,50.149,2026-06-18T06:08:48Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-2356,47896,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,79.033,2026-06-17T23:34:49Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-3297,50159,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:05:00+00:00,300,88.099,2026-06-17T07:41:29Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-3289,50173,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,38.942,2026-06-18T02:40:02Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-3283,50179,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:01:00+00:00,60,28.556,2026-06-18T02:40:46Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-3267,50197,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,44.213,2026-06-18T02:42:05Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
